# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Scepter70/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Scepter70/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "datasets", "duckdb"], check=True)

from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))


*Run the cell above first. Paste its output to Claude before continuing — the exact table/column names determine how sections 1-4 below get filled in.*

In [24]:
# EXPLORATION ONLY — run this once to see what's actually in the warehouse release,
# then send Claude the printed output before filling in sections 1-4 below.
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print("Files in the dataset repo:")
for f in files:
    print(" ", f)


Files in the dataset repo:
  .gitattributes
  README.md
  dim_clients.parquet
  dim_content.parquet
  fact_content_daily_performance/month=2025-01/data_0.parquet
  fact_content_daily_performance/month=2025-02/data_0.parquet
  fact_content_daily_performance/month=2025-03/data_0.parquet
  fact_content_daily_performance/month=2025-04/data_0.parquet
  fact_content_daily_performance/month=2025-05/data_0.parquet
  fact_content_daily_performance/month=2025-06/data_0.parquet
  fact_content_daily_performance/month=2025-07/data_0.parquet
  fact_content_daily_performance/month=2025-08/data_0.parquet
  fact_content_daily_performance/month=2025-09/data_0.parquet
  fact_content_daily_performance/month=2025-10/data_0.parquet
  fact_content_daily_performance/month=2025-11/data_0.parquet
  fact_content_daily_performance/month=2025-12/data_0.parquet
  fact_content_daily_performance/month=2026-01/data_0.parquet
  fact_content_daily_performance/month=2026-02/data_0.parquet
  fact_content_daily_performance

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row (after aggregation) = one page** (a `client_hash_id` + `content_hash_id` pair), summarized over **one calendar month** (I'm using `month=2026-03` as my mid-panel month, not the `_sample` file, which is only the final month and shouldn't be used to develop label logic).

The raw fact table itself is finer-grained than this — one row = one page, for one client, on one day. My lane aggregates that daily grain up to a monthly page-level summary, which is the unit I actually need for a refresh-opportunity score.

In [25]:
import pandas as pd

fact = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet")
dim_content = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet")
dim_clients = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet")

print("=== fact_content_daily_performance (month=2026-03) ===")
print(f"Rows: {len(fact):,}")
print(fact.dtypes)
print(fact.head(3))

print("\n=== dim_content ===")
print(f"Rows: {len(dim_content):,}")
print(dim_content.dtypes)

print("\n=== dim_clients ===")
print(f"Rows: {len(dim_clients):,}")
print(dim_clients.dtypes)

print("fact_content_daily_performance columns:")
for col, dtype in fact.dtypes.items():
    print(f"  {col}: {dtype}")

=== fact_content_daily_performance (month=2026-03) ===
Rows: 9,841,378
report_date                   object
client_hash_id                object
content_hash_id               object
client_has_gsc                  bool
client_has_ga4                  bool
gsc_data_available              bool
ga4_data_available            object
gsc_impressions                int64
gsc_clicks                     int64
gsc_sum_position               int64
gsc_avg_position             float64
ga4_pageviews                float64
ga4_sessions                 float64
ga4_users                    float64
ga4_engaged_sessions         float64
ga4_total_engagement_sec     float64
sessions_organic             float64
sessions_direct              float64
sessions_referral            float64
sessions_social              float64
sessions_paid                float64
sessions_ai                  float64
ai_chatgpt                   float64
ai_perplexity                float64
ai_gemini                    float64
ai_c

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (knowable at decision time):** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `search_volume`, `competition`, `word_count`, `backlinks`, `last_optimized_date` (recency signal).

**Label/proxy:** no ready-made trend label exists in this table — it has to be computed by comparing this month's aggregates against a prior month's for the same page. That comparison isn't done yet; it's next week's work.

**Context (join keys, not features):** `client_hash_id`, `content_hash_id`, `report_date`/`month`.

**Excluded:** `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` — these AI-referral columns are almost entirely null in this month's data (confirmed in my query below), so they're not usable yet. Also excluding any row where `is_deleted = True` — a deleted page isn't a real refresh candidate.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
import duckdb
q1 = "SELECT COUNT(*) AS total_rows, COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || report_date) AS distinct_keys FROM fact"
grain_check = duckdb.sql(q1).df()
print("Grain check:")
print(grain_check)
q2 = "SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM fact"
span_check = duckdb.sql(q2).df()
print("Row count and date span:")
print(span_check)
q3 = "SELECT COUNT(*) AS available_rows FROM fact WHERE gsc_data_available IS TRUE"
availability_check = duckdb.sql(q3).df()
total_rows = len(fact)
avail = availability_check['available_rows'][0]
print("Availability:", avail, "of", total_rows, "rows have gsc_data_available IS TRUE")
print("ai_chatgpt null rate:", fact['ai_chatgpt'].isna().mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check:
   total_rows  distinct_keys
0     9841378        9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Row count and date span:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31
Availability: 3611061 of 9841378 rows have gsc_data_available IS TRUE
ai_chatgpt null rate: 0.30673966592889734


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

A single month in isolation can't show trend or decline — that requires joining at least two month partitions for the same page, which this notebook doesn't do yet. Client history is also unbalanced: `dim_clients.gsc_data_start` and `ga4_data_start` differ per client, so some client-months are GSC-only or missing GA4 entirely. Only 36.7% of rows this month have `gsc_data_available IS TRUE` — meaning nearly two-thirds of rows can't be trusted for GSC-based features without checking that flag first. The AI-referral columns are partially null (about 31% for `ai_chatgpt`), so any AI-referral signal needs more validation before use. Finally, this is real (anonymized) client production data — per the FlyRank data-use terms, no client-identifying values can appear in this notebook's output, only aggregate counts.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.